<a href="https://colab.research.google.com/github/huyenpiano1711/BTQT2_Telco_dbt_BigQuery/blob/main/telco_bronze_to_silver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# telco_bronze_to_silver.ipynb
**Dự án:** Telco Customer Churn – Medallion (Bronze → Silver) với BigQuery + dbt

| Phần | Nội dung | Phụ trách |
|---|---|---|
| 1 | Khảo sát & profiling dataset (21 cột) | Phân tích dataset & Bronze |
| 2 | Nạp CSV vào `bronze_telco.customers_raw` | Phân tích dataset & Bronze |
| 3 | dbt staging / Silver (`stg_telco_customers`, `customers_cleaned`) | Thành viên dbt |

Quy tắc đặt tên: file gốc `telco_customer_churn.csv` · Bronze `bronze_telco.customers_raw` · Silver `silver_telco.customers_cleaned`.

## 0. Cấu hình & xác thực

In [ ]:
PROJECT_ID  = "btqt2-telco-dbt"
BQ_LOCATION = "asia-southeast1"       # giữ cùng location cho Bronze và Silver
BRONZE_DATASET = "bronze_telco"
BRONZE_TABLE   = "customers_raw"
SOURCE_FILE    = "telco_customer_churn.csv"

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
print("Đã kết nối BigQuery:", bq.project)

Đã kết nối BigQuery: btqt2-telco-dbt


## 1. Nạp file CSV gốc
Upload file `WA_Fn-UseC_-Telco-Customer-Churn.csv` (tên gốc từ Kaggle/IBM); ô dưới sẽ đổi tên về `telco_customer_churn.csv` theo quy ước.

In [ ]:
import os, pandas as pd
from google.colab import files

if not os.path.exists(SOURCE_FILE):
    up = files.upload()                       # chọn file CSV
    name = next(iter(up))
    os.rename(name, SOURCE_FILE)
print("Kích thước file:", os.path.getsize(SOURCE_FILE), "bytes")

# Bronze = dữ liệu thô: đọc TẤT CẢ cột dạng chuỗi, KHÔNG biến chuỗi rỗng thành NaN
df = pd.read_csv(SOURCE_FILE, dtype=str, keep_default_na=False)
print(df.shape)
df.head()

Saving telco_customer_churn.csv to telco_customer_churn.csv
Kích thước file: 977501 bytes
(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes


## 2. Khảo sát cấu trúc 21 cột

In [ ]:
import numpy as np
raw = pd.read_csv(SOURCE_FILE)               # đọc tự suy kiểu để thấy kiểu pandas suy ra
summary = pd.DataFrame({
    "kieu_pandas_suy_ra": raw.dtypes.astype(str),
    "so_gia_tri_duy_nhat": df.nunique(),
    "vi_du": df.iloc[0],
})
summary

,kieu_pandas_suy_ra,so_gia_tri_duy_nhat,vi_du
customerID,object,7043,7590-VHVEG
gender,object,2,Female
SeniorCitizen,int64,2,0
Partner,object,2,Yes
Dependents,object,2,No
tenure,int64,73,1
PhoneService,object,2,No
MultipleLines,object,3,No phone service
InternetService,object,3,DSL
OnlineSecurity,object,3,No


In [ ]:
# Phân bố các cột phân loại (<= 6 giá trị)
for c in df.columns:
    if df[c].nunique() <= 6:
        print(f"{c:18s}", df[c].value_counts().to_dict())

gender             {'Male': 3555, 'Female': 3488}
SeniorCitizen      {'0': 5901, '1': 1142}
Partner            {'No': 3641, 'Yes': 3402}
Dependents         {'No': 4933, 'Yes': 2110}
PhoneService       {'Yes': 6361, 'No': 682}
MultipleLines      {'No': 3390, 'Yes': 2971, 'No phone service': 682}
InternetService    {'Fiber optic': 3096, 'DSL': 2421, 'No': 1526}
OnlineSecurity     {'No': 3498, 'Yes': 2019, 'No internet service': 1526}
OnlineBackup       {'No': 3088, 'Yes': 2429, 'No internet service': 1526}
DeviceProtection   {'No': 3095, 'Yes': 2422, 'No internet service': 1526}
TechSupport        {'No': 3473, 'Yes': 2044, 'No internet service': 1526}
StreamingTV        {'No': 2810, 'Yes': 2707, 'No internet service': 1526}
StreamingMovies    {'No': 2785, 'Yes': 2732, 'No internet service': 1526}
Contract           {'Month-to-month': 3875, 'Two year': 1695, 'One year': 1473}
PaperlessBilling   {'Yes': 4171, 'No': 2872}
PaymentMethod      {'Electronic check': 2365, 'Mailed check': 1612, '

In [ ]:
# Thống kê các cột số
num = df[["SeniorCitizen","tenure","MonthlyCharges","TotalCharges"]].apply(pd.to_numeric, errors="coerce")
num.describe().round(2)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00,7032.00
mean,0.16,32.37,64.76,2283.30
std,0.37,24.56,30.09,2266.77
min,0.00,0.00,18.25,18.80
25%,0.00,9.00,35.50,401.45
50%,0.00,29.00,70.35,1397.48
75%,0.00,55.00,89.85,3794.74
max,1.00,72.00,118.75,8684.80


## 3. Kiểm tra chất lượng dữ liệu

In [ ]:
# 3.1 Thiếu: NULL thật + chuỗi rỗng/khoảng trắng + token 'NA/null/...'
tokens = {"", "na", "n/a", "null", "nan", "none"}
blank = df.apply(lambda s: s.str.strip().str.lower().isin(tokens)).sum()
print("Cột có giá trị thiếu/trống:")
print(blank[blank > 0] if (blank > 0).any() else "Không có")

# 3.2 Trùng
print("\nDòng trùng hoàn toàn      :", df.duplicated().sum())
print("customerID trùng          :", df.customerID.duplicated().sum())
print("customerID sai định dạng  :", (~df.customerID.str.match(r"^\d{4}-[A-Z]{5}$")).sum())

# 3.3 Khoảng trắng đầu/cuối
ws = df.apply(lambda s: (s != s.str.strip()).sum())
print("\nCột có khoảng trắng thừa:", ws[ws > 0].to_dict())

Cột có giá trị thiếu/trống:
TotalCharges    11
dtype: int64

Dòng trùng hoàn toàn      : 0
customerID trùng          : 0
customerID sai định dạng  : 0

Cột có khoảng trắng thừa: {'TotalCharges': 11}


In [ ]:
# 3.4 Sai kiểu: cột nào không ép được sang số?
for c in ["SeniorCitizen","tenure","MonthlyCharges","TotalCharges"]:
    bad = pd.to_numeric(df[c], errors="coerce").isna().sum()
    print(f"{c:15s} không ép được sang số: {bad}")

bad_tc = df[pd.to_numeric(df.TotalCharges, errors="coerce").isna()]
print("\nChi tiết các dòng TotalCharges lỗi:")
display(bad_tc[["customerID","tenure","MonthlyCharges","TotalCharges","Contract","Churn"]]
        .assign(TotalCharges=lambda d: d.TotalCharges.map(repr)))

SeniorCitizen   không ép được sang số: 0
tenure          không ép được sang số: 0
MonthlyCharges  không ép được sang số: 0
TotalCharges    không ép được sang số: 11

Chi tiết các dòng TotalCharges lỗi:


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,' ',Two year,No
753,3115-CZMZD,0,20.25,' ',Two year,No
936,5709-LVOEQ,0,80.85,' ',Two year,No
1082,4367-NUYAO,0,25.75,' ',Two year,No
1340,1371-DWPAZ,0,56.05,' ',Two year,No
3331,7644-OMVMY,0,19.85,' ',Two year,No
3826,3213-VVOLG,0,25.35,' ',Two year,No
4380,2520-SGTTA,0,20,' ',Two year,No
5218,2923-ARZLG,0,19.7,' ',One year,No
6670,4075-WKNIU,0,73.35,' ',Two year,No


In [ ]:
# 3.5 Kiểm tra logic nghiệp vụ
internet_cols = ["OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]
ni = df.InternetService == "No"
print("InternetService='No'       :", ni.sum())
for c in internet_cols:
    print(f"  {c:17s} 'No internet service' = {(df[c]=='No internet service').sum()}  | mâu thuẫn = {((~ni)&(df[c]=='No internet service')).sum()}")
print("PhoneService='No'          :", (df.PhoneService=='No').sum(),
      "| MultipleLines='No phone service' =", (df.MultipleLines=='No phone service').sum())

t = pd.to_numeric(df.TotalCharges, errors="coerce")
m = pd.to_numeric(df.MonthlyCharges); n = pd.to_numeric(df.tenure)
ratio = t / (m * n).replace(0, np.nan)
print("\ntenure = 0                 :", (n==0).sum())
print("MonthlyCharges <= 0        :", (m<=0).sum())
print("TotalCharges < MonthlyCharges (tenure>0):", ((t<m)&(n>0)).sum())
print("TotalCharges / (Monthly*tenure): min={:.2f}, max={:.2f}, std={:.3f}".format(ratio.min(), ratio.max(), ratio.std()))
print("\nTỷ lệ Churn:", (df.Churn=='Yes').mean().round(4), f"({(df.Churn=='Yes').sum()}/{len(df)})")

InternetService='No'       : 1526
  OnlineSecurity    'No internet service' = 1526  | mâu thuẫn = 0
  OnlineBackup      'No internet service' = 1526  | mâu thuẫn = 0
  DeviceProtection  'No internet service' = 1526  | mâu thuẫn = 0
  TechSupport       'No internet service' = 1526  | mâu thuẫn = 0
  StreamingTV       'No internet service' = 1526  | mâu thuẫn = 0
  StreamingMovies   'No internet service' = 1526  | mâu thuẫn = 0
PhoneService='No'          : 682 | MultipleLines='No phone service' = 682

tenure = 0                 : 11
MonthlyCharges <= 0        : 0
TotalCharges < MonthlyCharges (tenure>0): 0
TotalCharges / (Monthly*tenure): min=0.69, max=1.57, std=0.051

Tỷ lệ Churn: 0.2654 (1869/7043)


## 4. Bảng vấn đề dữ liệu

In [ ]:
issues = pd.DataFrame([
 ["P1","TotalCharges","Sai kiểu / thiếu ngầm",
  "11 dòng chứa 1 ký tự khoảng trắng ' ' nên cả cột bị đọc thành chuỗi; CAST sang số sẽ lỗi. Cả 11 dòng đều có tenure=0, Churn=No (10 Two year, 1 One year) – khách vừa đăng ký, chưa phát sinh hóa đơn.",
  11,"Cao","Silver: TRIM → SAFE_CAST AS NUMERIC → NULL; thêm cờ is_new_customer (tenure=0). Nhóm chốt NULL hay 0."],
 ["P2","SeniorCitizen","Mã hóa không nhất quán",
  "Là 0/1 trong khi mọi cột nhị phân khác dùng Yes/No.",
  7043,"Trung bình","Ép BOOLEAN (is_senior_citizen)."],
 ["P3","MultipleLines","Giá trị dư thừa",
  "682 dòng 'No phone service' trùng thông tin với PhoneService='No'.",
  682,"Thấp","Chuẩn hóa về 'No' (giữ PhoneService để bảo toàn thông tin)."],
 ["P4","OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies","Giá trị dư thừa",
  "1.526 dòng 'No internet service' ở cả 6 cột, trùng hoàn toàn với InternetService='No'. Không có dòng mâu thuẫn.",
  1526,"Thấp","Chuẩn hóa về 'No' rồi ép BOOLEAN."],
 ["P5","Partner, Dependents, PhoneService, PaperlessBilling, Churn (+ P3, P4)","Sai kiểu đích",
  "Cột nhị phân lưu dạng chuỗi 'Yes'/'No'.",
  7043,"Thấp","Ép BOOLEAN (churn → is_churned)."],
 ["P6","tenure, MonthlyCharges, SeniorCitizen","Kiểu dữ liệu thô",
  "Toàn bộ CSV là text; cần ép tenure→INT64, MonthlyCharges→NUMERIC.",
  7043,"Thấp","Ép kiểu ở stg_telco_customers."],
 ["P7","Toàn bộ 21 cột","Quy ước đặt tên",
  "Tên cột lẫn camelCase/PascalCase/lowercase (customerID, gender, SeniorCitizen, tenure…).",
  21,"Thấp","Đổi snake_case ở staging (customer_id, senior_citizen, …)."],
 ["P8","TotalCharges vs MonthlyCharges×tenure","Nhất quán số học (lưu ý)",
  "Tỷ lệ Total/(Monthly×tenure) dao động 0,69–1,57 (độ lệch chuẩn ≈0,05); phản ánh đổi gói cước. Không có dòng Total < Monthly khi tenure>0.",
  2,"Thông tin","Không sửa; chỉ ghi nhận. (2 dòng lệch >50%.)"],
 ["P9","Churn","Mất cân bằng lớp (lưu ý)",
  "1.869 Yes (26,54%) / 5.174 No (73,46%).",
  1869,"Thông tin","Ghi nhận cho phần phân tích/mô hình sau Silver."],
], columns=["Mã","Cột","Loại vấn đề","Mô tả","Số dòng ảnh hưởng","Mức độ","Hướng xử lý (Silver)"])
pd.set_option("display.max_colwidth", 120)
issues

,Mã,Cột,Loại vấn đề,Mô tả,Số dòng ảnh hưởng,Mức độ,Hướng xử lý (Silver)
0,P1,TotalCharges,Sai kiểu / thiếu ngầm,11 dòng chứa 1 ký tự khoảng trắng ' ' nên cả cột bị đọc thành chuỗi; CAST sang số sẽ lỗi. Cả 11 dòng đều có tenure=0...,11,Cao,Silver: TRIM → SAFE_CAST AS NUMERIC → NULL; thêm cờ is_new_customer (tenure=0). Nhóm chốt NULL hay 0.
1,P2,SeniorCitizen,Mã hóa không nhất quán,Là 0/1 trong khi mọi cột nhị phân khác dùng Yes/No.,7043,Trung bình,Ép BOOLEAN (is_senior_citizen).
2,P3,MultipleLines,Giá trị dư thừa,682 dòng 'No phone service' trùng thông tin với PhoneService='No'.,682,Thấp,Chuẩn hóa về 'No' (giữ PhoneService để bảo toàn thông tin).
3,P4,"OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies",Giá trị dư thừa,"1.526 dòng 'No internet service' ở cả 6 cột, trùng hoàn toàn với InternetService='No'. Không có dòng mâu thuẫn.",1526,Thấp,Chuẩn hóa về 'No' rồi ép BOOLEAN.
4,P5,"Partner, Dependents, PhoneService, PaperlessBilling, Churn (+ P3, P4)",Sai kiểu đích,Cột nhị phân lưu dạng chuỗi 'Yes'/'No'.,7043,Thấp,Ép BOOLEAN (churn → is_churned).
5,P6,"tenure, MonthlyCharges, SeniorCitizen",Kiểu dữ liệu thô,"Toàn bộ CSV là text; cần ép tenure→INT64, MonthlyCharges→NUMERIC.",7043,Thấp,Ép kiểu ở stg_telco_customers.
6,P7,Toàn bộ 21 cột,Quy ước đặt tên,"Tên cột lẫn camelCase/PascalCase/lowercase (customerID, gender, SeniorCitizen, tenure…).",21,Thấp,"Đổi snake_case ở staging (customer_id, senior_citizen, …)."
7,P8,TotalCharges vs MonthlyCharges×tenure,Nhất quán số học (lưu ý),"Tỷ lệ Total/(Monthly×tenure) dao động 0,69–1,57 (độ lệch chuẩn ≈0,05); phản ánh đổi gói cước. Không có dòng Total < ...",2,Thông tin,Không sửa; chỉ ghi nhận. (2 dòng lệch >50%.)
8,P9,Churn,Mất cân bằng lớp (lưu ý),"1.869 Yes (26,54%) / 5.174 No (73,46%).",1869,Thông tin,Ghi nhận cho phần phân tích/mô hình sau Silver.


In [ ]:
# Xác nhận không có: NULL thật, dòng trùng, customerID trùng
assert df.customerID.is_unique and df.duplicated().sum() == 0
assert len(df) == 7043 and df.shape[1] == 21
print("OK: 7.043 dòng, 21 cột, không trùng.")

## 5. Nạp vào BigQuery Bronze (`bronze_telco.customers_raw`)
Nguyên tắc Bronze: **giữ nguyên dữ liệu gốc** (mọi cột STRING, giữ nguyên `' '` của TotalCharges, giữ nguyên tên cột) + 2 cột metadata để truy vết. Mọi làm sạch/ép kiểu thực hiện ở dbt.

In [ ]:
from google.cloud.exceptions import NotFound

dataset_id = f"{PROJECT_ID}.{BRONZE_DATASET}"
ds = bigquery.Dataset(dataset_id); ds.location = BQ_LOCATION
bq.create_dataset(ds, exists_ok=True)

# Schema tường minh: 21 cột STRING + metadata
schema = [bigquery.SchemaField(c, "STRING") for c in df.columns] + [
    bigquery.SchemaField("_ingested_at", "TIMESTAMP"),
    bigquery.SchemaField("_source_file", "STRING"),
]

bronze_df = df.copy()
bronze_df["_ingested_at"] = pd.Timestamp.now(tz="UTC")
bronze_df["_source_file"] = SOURCE_FILE

table_id = f"{dataset_id}.{BRONZE_TABLE}"
job = bq.load_table_from_dataframe(
    bronze_df, table_id,
    job_config=bigquery.LoadJobConfig(schema=schema, write_disposition="WRITE_TRUNCATE"),  # chạy lại không bị nhân đôi
)
job.result()
print("Đã nạp", bq.get_table(table_id).num_rows, "dòng vào", table_id)

NotFound: 404 POST https://bigquery.googleapis.com/bigquery/v2/projects/btqt2-telco-dbt/datasets?prettyPrint=false: Project btqt2-telco-dbt is not found. Make sure it references valid GCP project that hasn't been deleted.; Project id: btqt2-telco-dbt

### Kiểm tra sau nạp

In [ ]:
q = f"""
SELECT
  COUNT(*)                                   AS so_dong,
  COUNT(DISTINCT customerID)                 AS so_customer_id,
  COUNTIF(TRIM(TotalCharges) = '')           AS totalcharges_trong,
  COUNTIF(Churn = 'Yes')                     AS so_churn
FROM `{table_id}`
"""
res = bq.query(q).to_dataframe()
display(res)
assert res.so_dong[0] == 7043 and res.so_customer_id[0] == 7043 and res.totalcharges_trong[0] == 11 and res.so_churn[0] == 1869
print("Bronze khớp file nguồn.")
bq.get_table(table_id).schema

## 6. (Tiếp theo) dbt – Staging & Silver
Phần này do thành viên dbt hoàn thiện: project `dbt_telco`, model `stg_telco_customers.sql` → `customers_cleaned.sql` (ghi vào `silver_telco.customers_cleaned`), kiểm thử trong `schema.yml`. Danh sách quy tắc xử lý đã nêu ở bảng vấn đề (mục 4).